# Regression models

In [ ]:
# import packages
#Random state
a=19 # 42, 13 101 19

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import pyplot

from numpy import mean
from numpy import std
from numpy import absolute
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.linear_model import Lasso

#Speicherpfad muss individuell angepasst werden Excel aus ordner Code Verwenden
df = pd.read_excel ('../data/Daten_Siburg.xlsx')



# drop irrelevant column 
df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)
df.drop(labels='Nr', axis=1, inplace=True)

def numCol(df):
    df_numeric = df.select_dtypes(include=[np.number])
    cols = df_numeric.columns.values
    return cols

# apply function
numeric_cols= numCol(df)

#replacement
df.replace('fcm,cyl', 'fcmcyl')

# drop all column containing only one unique value
# find single unique value column
col_sv=df.columns[df.nunique() == 1]

# replacing invalid values
df.replace('-', np.nan, inplace=True)

df.drop(['Bez Vers.Ber', 'Forscher', 'Platte', 'Last','Umfang', 'Stütze', 'c2', 'Esm', 'c1','dg','h', 'fym'], axis = 1, inplace=True)
print(df)


In [ ]:
#import MinMaxScaler from sklearn library
from sklearn.preprocessing import MinMaxScaler

# select the properties that should be scaled
cols=['Lasteinleitung', 'd', 'Fläche', 'rho_l', 'fcmcyl', 'V_test']

# Create Scaler object
scaler = MinMaxScaler(feature_range=(0.000001, 1))
# fit scaler to data
scaler.fit(df[cols])
#scale chosen properties
df_scaled = pd.DataFrame(scaler.transform(df[cols]), columns=cols)

# inspect distribution
print(df_scaled)

In [ ]:
df_scaled.hist(bins=50, figsize=(9,9), color='grey')
plt.savefig('Histogramm.png')

In [ ]:
df_scaled.boxplot(figsize=(10,5))
plt.savefig('Boxplot.png')

In [ ]:
data = df_scaled.values
X, y = data[:, :-1], data[:, -1]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=a)
df_train, df_test = train_test_split(df_scaled, test_size=0.3,random_state=a)

## Linear Lasso

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

In [ ]:
# consider to EX3 and investigate feautures
import statsmodels.api as sm
import statsmodels.formula.api as smf

mlr_1 = smf.ols(formula='V_test ~  d + Fläche + rho_l + fcmcyl' , data=df_train).fit()

print(mlr_1.summary())

high p-value (>0.05): low significance of the feature
low p-value (<0.025): high significance of the feature
so fym is removed further Fläche is also removed since higher variance than Umfang

In [ ]:
import seaborn as sns
corrMatrix=df_train .corr()
sns.heatmap(corrMatrix, annot=True)

In [ ]:
from statistics import variance
print(df. var()['Fläche'])
#print(df. var()['Umfang'])

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def VIF(df, columns):
    
    values = sm.add_constant(df[columns]).values  
    # the dataframe passed to VIF must include the intercept term -> add column
    num_columns = len(columns)+1
    vif = [variance_inflation_factor(values, i) for i in range(num_columns)]
    
    return pd.Series(vif[1:], index=columns)

#calculate VIF Value
X_cols= df_train .columns.drop('V_test')
VIF(df_train , X_cols)

In [ ]:
model_0 = smf.ols(formula='V_test ~ d + Fläche + rho_l + fcmcyl' , data=df_train).fit()

print(model_0.summary())

### Evaluation of the linear model

In [ ]:
# define Out-of-sample R2 function
def OSR2(model_0, df_train, df_test, dependent_var):   
    
    y_test = df_test[dependent_var]
    y_pred = model_0.predict(df_test)
    SSE = np.sum((y_test - y_pred)**2)
    SST = np.sum((y_test - np.mean(df_train[dependent_var]))**2)    
    
    return 1 - SSE/SST

In [ ]:
print('OSR2:', round(OSR2(model_0, df_train , df_test, 'V_test'), 5))

In [ ]:
#why does this not work?!
yhat_0=model_0.predict(df_test)
error_0=np.sum((yhat_0-y_test)**2)/len(yhat_0)
print(error_0)

yhat_0t=model_0.predict(df_train)

In [ ]:
plt.scatter(yhat_0  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Lin. Regression')
plt.grid(True,linestyle='-',color='0.75')
print('prediction underestimated meaning on the safe side')
print('MSE %f' %error_0)

plt.savefig('Linear_Model.png')

##  Nonlinear Regression

Trying to taking into account non linearity in the calculation, therefor d times fcmcyl is acconted which physically can also be justified

In [ ]:
model_1 = smf.ols(formula='V_test ~ d + Fläche + rho_l  + fcmcyl*d' , data=df_train).fit()

print(model_1.summary())

### Evaluation of the non-linar model

In [ ]:
print('OSR2:', round(OSR2(model_1, df_train , df_test, 'V_test'), 5))

In [ ]:
yhat_1=model_1.predict(df_test)
error_1=np.sum((yhat_1-y_test)**2)/len(yhat_1)
print(error_1)

yhat_1t=model_1.predict(df_train)

In [ ]:
plt.scatter(yhat_1  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Nonlinear Regression')
plt.grid(True,linestyle='-',color='0.75')
print('prediction underestimated meaning on the safe side')
print('MSE %f' %error_1)

## Lasso

In [ ]:
from sklearn.linear_model import LassoCV
from numpy import arange
## define model evaluation method
#cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
# define model
#model = LassoCV(alphas=arange(0, 0.01, 0.001), cv=cv, n_jobs=-1)
# fit model
#model.fit(X_train, y_train)
# summarize chosen configuration
#print('alpha: %f' % model.alpha_)

In [ ]:
# grid search hyperparameters for lasso regression
from sklearn.model_selection import GridSearchCV
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
#define model
model=Lasso()
# define grid
grid = dict()
grid['alpha'] = arange(0, 0.001, 0.0001)
# define search
search = GridSearchCV(model, grid, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# perform the search
results = search.fit(X_train, y_train)
# summarize
print('MAE: %.3f' % results.best_score_)
print('Config: %s' % results.best_params_)

In [ ]:
model_2 = Lasso(alpha=0.0007)
model_2.fit(X_train, y_train)
yhat_2=model_2.predict(X_test)
error_2=np.sum((yhat_2-y_test)**2)/len(yhat_2)
print(error_2)


yhat_2t=model_2.predict(X_train)

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def VIF(df, columns):
    
    values = sm.add_constant(df[columns]).values  
    # the dataframe passed to VIF must include the intercept term -> add column
    num_columns = len(columns)+1
    vif = [variance_inflation_factor(values, i) for i in range(num_columns)]
    
    return pd.Series(vif[1:], index=columns)

In [ ]:
#calculate VIF Value
X_cols= df.columns.drop('V_test')
VIF(df , X_cols)

In [ ]:
from sklearn.datasets import make_regression

# get importance
importance = model_2.coef_
# summarize feature importance
for i,v in enumerate(importance):
	print('Feature: %0d, Score: %.4f' % (i,v))
# plot feature importance
pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()

In [ ]:
plt.scatter(yhat_2  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Lasso Regression')
plt.grid(True,linestyle='-',color='0.75')
print('prediction underestimated meaning on the safe side')
print('MSE %f' %error_2)

plt.savefig('Lasso_Model.png')

In [ ]:
import shap
print("Model coefficients:\n")
for i in range(X_test.shape[1]):
    print(df_test.columns[i], "=", model_2.coef_[i].round(4))

In [ ]:
explainer = shap.Explainer(model_2.predict, X_train)
shap_values = explainer(X_train)
sample_ind =33
shap.plots.waterfall(shap_values[sample_ind], max_display=14)

shap.partial_dependence_plot(
    "Feature 0", model_2.predict, X_test, model_expected_value=True,
    feature_expected_value=True, ice=False,
    shap_values=shap_values[sample_ind:sample_ind+1,:])

In [ ]:
shap.summary_plot(shap_values, X_train)

 shap.dependence_plot("d", shap_values, df_train)
#doesn t work 


#shap.force_plot(explainer.expected_value, shap_values, X_train)

In [ ]:
#conda install -c conda-forge pyGAM
#conda install -c interpretml interpret

In [ ]:
#https://www.kaggle.com/prashant111/explain-your-model-predictions-with-shapley-values

# Ridge Regression

In [ ]:
from sklearn.linear_model import RidgeCV
# define model
model = RidgeCV(alphas=arange(0.1, 1, 0.1), cv=cv, scoring='neg_mean_absolute_error')
# fit model
model.fit(X_train, y_train)
# summarize chosen configuration
print('alpha: %f' % model.alpha_)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
model_3 = Ridge(alpha=0.55)
model_3.fit(X_test, y_test)
yhat_3=model_3.predict(X_test)
error_3=np.sum((yhat_3-y_test)**2)/len(yhat_3)
print(error_3)

yhat_3t=model_3.predict(X_train)

In [ ]:
# get importance
importance = model_3.coef_
# summarize feature importance
for i,v in enumerate(importance):
	print('Feature: %0d, Score: %.4f' % (i,v))
# plot feature importance
pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()

In [ ]:
plt.scatter(yhat_3  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Ridge Regression')
plt.grid(True,linestyle='-',color='0.75')
print('prediction underestimated meaning on the safe side')
print('MSE %f' %error_3)

plt.savefig('Ridge_Model.png')

# Elastic Net

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
# define model
model = ElasticNet()
# define grid
grid = dict()
grid['alpha'] = arange(0.001,0.01,0.001)
grid['l1_ratio'] = arange(0, 0.3, 0.05)
# define search
search = GridSearchCV(model, grid, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# perform the search
results = search.fit(X_train, y_train)
# summarize
print('MAE: %.3f' % results.best_score_)
print('Config: %s' % results.best_params_)

In [ ]:
model_4 = ElasticNet(alpha=0.003, l1_ratio=0.05)
# fit model
model_4.fit(X_train, y_train)
yhat_4=model_4.predict(X_test)
error_4=np.sum((yhat_4-y_test)**2)/len(yhat_4)
print(error_4)

yhat_4t=model_4.predict(X_train)

In [ ]:
# get importance
importance = model_4.coef_
# summarize feature importance
for i,v in enumerate(importance):
	print('Feature: %0d, Score: %.4f' % (i,v))
# plot feature importance
pyplot.bar([x for x in range(len(importance))], importance)
pyplot.show()

In [ ]:
plt.scatter(yhat_4  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Elastic Net Regression')
plt.grid(True,linestyle='-',color='0.75')
print('prediction underestimated meaning on the safe side')
print('MSE %f' %error_4)

plt.savefig('Elastic_Net_Model.png')

# Evaluation and comparision

In [ ]:
#performing on the train data ideal Data
train_error_0 = np.sum((yhat_0t-y_train)**2)/len(yhat_0t)
train_error_1 = np.sum((yhat_1t-y_train)**2)/len(yhat_1t)
train_error_2 = np.sum((yhat_2t-y_train)**2)/len(yhat_2t)
train_error_3 = np.sum((yhat_3t-y_train)**2)/len(yhat_3t)
train_error_4 = np.sum((yhat_4t-y_train)**2)/len(yhat_4t)
print('Linar Regression: %.4f' % train_error_0)
print('Non_linar Regression: %.4f' % train_error_1)
print('Lasso: %.4f' % train_error_2)
print('Ridge: %.4f' % train_error_3)
print('Elastic Net: %.4f' % train_error_4)

In [ ]:
#performing on the test Data MSE
print('Lin. Reg.: %.4f' % error_0)
print('Nonlin. Reg.: %.4f' % error_1)
print('Lasso: %.4f' % error_2)
print('Ridge: %.4f' % error_3)
print('Elastic Net: %.4f' % error_4)

In [ ]:
#Test values
MAE_0=np.sum(abs(yhat_0-y_test))/len(yhat_0)
MAE_1=np.sum(abs(yhat_1-y_test))/len(yhat_1)
MAE_2=np.sum(abs(yhat_2-y_test))/len(yhat_2)
MAE_3=np.sum(abs(yhat_3-y_test))/len(yhat_3)
MAE_4=np.sum(abs(yhat_4-y_test))/len(yhat_4)
print('MAE_0: %f' %MAE_0)
print('MAE_1: %f' %MAE_1)
print('MAE_2: %f' %MAE_2)
print('MAE_3: %f' %MAE_3)
print('MAE_4: %f' %MAE_4)

In [ ]:
#Train values
MAE_0t=np.sum(abs(yhat_0t-y_train))/len(yhat_0t)
MAE_1t=np.sum(abs(yhat_1t-y_train))/len(yhat_1t)
MAE_2t=np.sum(abs(yhat_2t-y_train))/len(yhat_2t)
MAE_3t=np.sum(abs(yhat_3t-y_train))/len(yhat_3t)
MAE_4t=np.sum(abs(yhat_4t-y_train))/len(yhat_4t)
print('MAE_0t: %f' %MAE_0t)
print('MAE_1t: %f' %MAE_1t)
print('MAE_2t: %f' %MAE_2t)
print('MAE_3t: %f' %MAE_3t)
print('MAE_4t: %f' %MAE_4t)

In [ ]:
# Plot data with a line and dots at the data points
# Plot data with a line and dots at the data points
train_MSE=[train_error_0, train_error_1,train_error_2,train_error_3,train_error_4]
test_MSE=[error_0, error_1,error_2,error_3,error_4]
test_MAE=[MAE_0,MAE_1,MAE_2,MAE_3,MAE_4]
train_MAE=[MAE_0t,MAE_1t,MAE_2t,MAE_3t,MAE_4t]
model=['Lin. Reg.', 'Nonlin. Reg.', 'Lasso', 'Ridge', 'Elastic Net' ]
plt.plot(model,train_MSE, label="$MSE, train$")
plt.plot(model,test_MSE, label="$MSE, test$")
plt.plot(model,test_MAE, label="$MAE, test$") 
plt.plot(model,train_MAE, label="$MAE, train$")
plt.legend()
    
# Attach labels and title 
plt.xlabel('$model-type$')
plt.ylabel('$error$')
plt.title(" Comparision MSE and MAE ")

plt.savefig('MSE_MAE_Overwiev.png')

plt.show()

In [ ]:
train_MSE=[train_error_2,train_error_3,train_error_4]
test_MSE=[error_2,error_3,error_4]
model=[ 'Lasso', 'Ridge', 'Elastic Net' ]
plt.plot(model,train_MSE, label="$MSE, train$")
plt.plot(model,test_MSE, label="$MSE, test$")
plt.legend()
    
# Attach labels and title 
plt.xlabel('$model-type$')
plt.ylabel('$error$')
plt.title(" Zoom in MSE")

plt.savefig('Zoom_MSE.png')

plt.show()

In [ ]:
test_MAE=[MAE_2,MAE_3,MAE_4]
train_MAE=[MAE_2t,MAE_3t,MAE_4t]

plt.plot(model,test_MAE, label="$MAE, test$") 
plt.plot(model,train_MAE, label="$MAE, train$")
plt.legend()
    
# Attach labels and title 
plt.xlabel('$model-type$')
plt.ylabel('$error$')
plt.title(" Zoom in MAE ")

plt.savefig('Zoom_MAE.png')

plt.show()

In [ ]:
#Random state 101
Config: {'alpha': 0.0009000000000000001}
alpha: 0.700000
Config: {'alpha': 0.003, 'l1_ratio': 0.1}
    
#Random state 42
Config: {'alpha': 0.0007}
alpha: 0.500000
Config: {'alpha': 0.003, 'l1_ratio': 0.0}
        
#Random state 13

Config: {'alpha': 0.0007}
alpha: 0.600000
Config: {'alpha': 0.003, 'l1_ratio': 0.0}
    
#Random state 19
Config: {'alpha': 0.0006000000000000001}
alpha: 0.500000
Config: {'alpha': 0.002, 'l1_ratio': 0.1}
                    
                    
#Decisicion on Parmaeters
Config: {'alpha': 0.0007}
alpha: 0.55
Config: {'alpha': 0.003, 'l1_ratio': 0.05}
